# Environment & Data Check

This notebook verifies the compute environment, dependencies, and validates that the dataset exists and is properly formatted.

**Outputs:**
- Environment info (GPU/CPU, Python version, key packages)
- Dataset statistics (size, label distribution)
- Sample data inspection

In [1]:
import sys
import json
from pathlib import Path
from datetime import datetime

import torch
import numpy as np
import pandas as pd
from transformers import __version__ as transformers_version
from sklearn import __version__ as sklearn_version

## 1. Environment Diagnostics

In [ ]:
# Detect hardware
print("="*60)
print("ENVIRONMENT CHECK")
print("="*60)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\n🔧 Device: {device.type.upper()}")

if torch.cuda.is_available():
    print(f"   GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"   GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"   CUDA Version: {torch.version.cuda}")
else:
    print(f"   CPU Count: {torch.get_num_threads()}")

print(f"\n📦 Key Packages:")
print(f"   Python: {sys.version.split()[0]}")
print(f"   PyTorch: {torch.__version__}")
print(f"   Transformers: {transformers_version}")
print(f"   scikit-learn: {sklearn_version}")
print(f"   NumPy: {np.__version__}")
print(f"   Pandas: {pd.__version__}")

## 2. Repository Structure Check

In [ ]:
# Find repo root and verify structure
def find_repo_root():
    """Find the sem-project root directory."""
    current = Path.cwd()
    while current != current.parent:
        if (current / "src").exists() and (current / "README.md").exists():
            return current
        current = current.parent
    raise RuntimeError("Could not find repo root. Are you inside the sem-project directory?")

try:
    REPO_ROOT = find_repo_root()
    print(f"\n📁 Repository Root: {REPO_ROOT}")
except RuntimeError as e:
    print(f"\n❌ Error: {e}")
    REPO_ROOT = Path.cwd()
    print(f"    Using current directory: {REPO_ROOT}")

# Check key directories
required_dirs = ["src", "src/data", "src/mcc", "src/doc_clf", "src/scripts", "src/checkpoints"]
print(f"\n✓ Required directories:")
for dir_name in required_dirs:
    dir_path = REPO_ROOT / dir_name
    status = "✓" if dir_path.exists() else "✗"
    print(f"  {status} {dir_name}")

## 3. Data File Validation

In [ ]:
# Check required data files
data_dir = REPO_ROOT / "src" / "data"
required_files = {
    "mcc_train.jsonl": "MCC training data",
    "mcc_dev.jsonl": "MCC validation data",
    "news_train.jsonl": "Document-level training data",
    "news_test.jsonl": "Document-level test data",
}

print(f"\n📄 Data Files ({data_dir}):")
file_stats = {}
for filename, description in required_files.items():
    filepath = data_dir / filename
    if filepath.exists():
        num_lines = sum(1 for _ in open(filepath, "r", encoding="utf-8"))
        size_mb = filepath.stat().st_size / 1e6
        print(f"  ✓ {filename:<20} {num_lines:>6} lines | {size_mb:>6.2f} MB | {description}")
        file_stats[filename] = num_lines
    else:
        print(f"  ✗ {filename:<20} MISSING!")
        file_stats[filename] = 0

## 4. MCC Dataset Inspection

In [ ]:
# Load and inspect MCC data
def load_jsonl(path):
    """Load JSONL file into list of dicts."""
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

mcc_train_data = load_jsonl(data_dir / "mcc_train.jsonl")
mcc_dev_data = load_jsonl(data_dir / "mcc_dev.jsonl")

print(f"\n📊 MCC Data Statistics:")
print(f"\n  Training set: {len(mcc_train_data)} sentences")

train_labels = [ex["label"].lower() for ex in mcc_train_data]
train_label_dist = pd.Series(train_labels).value_counts()
print(f"  Label distribution:")
for label, count in train_label_dist.items():
    pct = 100 * count / len(train_labels)
    print(f"    - {label:<10}: {count:>5} ({pct:>5.1f}%)")

print(f"\n  Dev set: {len(mcc_dev_data)} sentences")
dev_labels = [ex["label"].lower() for ex in mcc_dev_data]
dev_label_dist = pd.Series(dev_labels).value_counts()
for label, count in dev_label_dist.items():
    pct = 100 * count / len(dev_labels)
    print(f"    - {label:<10}: {count:>5} ({pct:>5.1f}%)")

In [ ]:
# Show sample MCC sentences
print(f"\n📝 Sample MCC Sentences (first 5 from training set):")
print("="*80)
for i, ex in enumerate(mcc_train_data[:5], 1):
    print(f"\n[{i}] Label: {ex['label'].upper()}")
    print(f"    Text: {ex['sentence'][:100]}..." if len(ex['sentence']) > 100 else f"    Text: {ex['sentence']}")

## 5. Document Classification Dataset Inspection

In [ ]:
# Load document-level data
news_train_data = load_jsonl(data_dir / "news_train.jsonl")
news_test_data = load_jsonl(data_dir / "news_test.jsonl")

print(f"\n📊 Document Classification Data Statistics:")
print(f"\n  Training set: {len(news_train_data)} documents")

train_doc_labels = [ex["label"].lower() for ex in news_train_data]
train_doc_dist = pd.Series(train_doc_labels).value_counts()
print(f"  Label distribution:")
for label, count in train_doc_dist.items():
    pct = 100 * count / len(train_doc_labels)
    print(f"    - {label:<10}: {count:>5} ({pct:>5.1f}%)")

print(f"\n  Test set: {len(news_test_data)} documents")
test_doc_labels = [ex["label"].lower() for ex in news_test_data]
test_doc_dist = pd.Series(test_doc_labels).value_counts()
for label, count in test_doc_dist.items():
    pct = 100 * count / len(test_doc_labels)
    print(f"    - {label:<10}: {count:>5} ({pct:>5.1f}%)")

In [ ]:
# Show document samples
print(f"\n📝 Sample Documents (first 2 from training set):")
print("="*80)
for i, ex in enumerate(news_train_data[:2], 1):
    label = ex.get("label", "unknown").upper()
    publisher = ex.get("publisher", "unknown")
    text = ex.get("text", "")
    print(f"\n[Document {i}]")
    print(f"  Label: {label}")
    print(f"  Publisher: {publisher}")
    text_preview = text[:150] + "..." if len(text) > 150 else text
    print(f"  Preview: {text_preview}")

## 6. Checkpoint Validation

In [ ]:
# Check for pre-trained checkpoints
checkpoint_dir = REPO_ROOT / "src" / "checkpoints"
print(f"\n🎯 Checkpoint Status ({checkpoint_dir}):")

if checkpoint_dir.exists():
    checkpoints = list(checkpoint_dir.glob("*.pt"))
    if checkpoints:
        for ckpt in sorted(checkpoints):
            size_mb = ckpt.stat().st_size / 1e6
            print(f"  ✓ {ckpt.name:<30} {size_mb:>8.2f} MB")
    else:
        print(f"  ⚠ No .pt checkpoint files found")
else:
    print(f"  ✗ Checkpoint directory does not exist")

## 7. Summary & Readiness

In [ ]:
# Final readiness check
print(f"\n" + "="*80)
print(f"READINESS SUMMARY")
print(f"="*80)

checks = {
    "GPU/CPU available": torch.cuda.is_available() or True,  # Always has CPU
    "Repo structure valid": all((REPO_ROOT / d).exists() for d in required_dirs),
    "All data files present": all(file_stats.values()),
    "MCC training data non-empty": len(mcc_train_data) > 0,
    "Document training data non-empty": len(news_train_data) > 0,
}

all_ready = all(checks.values())
for check_name, is_ok in checks.items():
    status = "✓" if is_ok else "✗"
    print(f"{status} {check_name}")

print(f"\n{'='*80}")
if all_ready:
    print(f"✓ All checks passed! Ready to run model comparisons.")
else:
    print(f"✗ Some checks failed. Please resolve missing files before proceeding.")

print(f"\nTimestamp: {datetime.now().isoformat()}")
print(f"Repo Root: {REPO_ROOT}")